In [ ]:

import cv2
import os
import mediapipe as mp
from mediapipe.tasks.python import vision


In [ ]:

landmarker = vision.HandLandmarker.create_from_model_path(
    model_path="hand_landmarker.task"
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
# 1. Data Preparation
IMG_SIZE = 128
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,   
    rotation_range=30,
    width_shift_range=0.2,   
    height_shift_range=0.2,
    zoom_range=0.15,
    shear_range=0.15,       
    brightness_range=[0.7, 1.3],
    fill_mode='nearest',
    validation_split=0.2
)
# 2. Load Data
train_gen = datagen.flow_from_directory(
    "dataset",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_gen = datagen.flow_from_directory(
    "dataset",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    "dataset",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)
# 3. Build the CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(256, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    
    Flatten(),
    Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.01)),
    Dropout(0.5),
    Dense(6, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=[early_stop])

loss, acc = model.evaluate(test_gen)
print(f"Test Accuracy: {acc*100:.2f}%")

model.save("gesture_cnn.h5")


In [ ]:
# 4.Real-time Gesture Recognition
import numpy as np


model = tf.keras.models.load_model("gesture_cnn.h5")
class_names = ['Down', 'Fist', 'Left', 'Palm', 'Right', 'Up']

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.flip(frame, 1)
    

    x1, y1, x2, y2 = 350, 100, 600, 350
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    

    roi = frame[y1:y2, x1:x2]
    img = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)


    predictions = model.predict(img, verbose=0)[0]
    idx = np.argmax(predictions)
    confidence = predictions[idx]

    if confidence > 0.7:
        gesture_name = class_names[idx].upper()
        
        cv2.rectangle(frame, (x1, y1-40), (x2, y1), (0, 255, 0), -1)
        cv2.putText(frame, gesture_name, (x1 + 10, y1 - 10), 
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, (0, 0, 0), 2)

    cv2.imshow("Gesture Recognition", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()